# Project 2 — Code Switching Dataset Collection
**Code Switching NLP | Code Saviours SI-26 | Humna Imran**

This notebook builds a labeled Roman Urdu + English code-switching dataset by starting from a **real, publicly available Roman Urdu dataset** (20,000+ organic sentences gathered from Twitter, Facebook, and e-commerce reviews) instead of manually scraping tweets one by one.

**Pipeline:**
1. Download the source dataset (`Smat26/Roman-Urdu-Dataset` on GitHub — a well-known, UCI-referenced Roman Urdu corpus)
2. Download a common-English wordlist (`google-10000-english`) to tell Urdu words from English words
3. Filter for sentences that are **genuinely code-switched** (a real natural mix of Urdu and English, not just one borrowed word, and not pure English)
4. Auto-label every word as `URD` / `ENG` / `MIX` using a dictionary lookup — the same core method used in published Roman-Urdu code-switching research (e.g. Ali & Sabir, *"An Attention Based Neural Network for Code Switching Detection: English & Roman Urdu"*, arXiv:2103.02252, who built their English side from the `dwyl/english-words` list)
5. Export the flat `dataset.csv` in the exact format the handout asks for, and push it to Hugging Face

> ⚠️ **Important — read before submitting:** the labels below are produced automatically by dictionary lookup, not by a human annotator. This is a legitimate bootstrapping method (real papers do the same), but it isn't perfect — short words that exist in both languages (e.g. "mil", "beta") are the main source of error. **Spot-check ~20–30 rows by hand (Step 5 below) before you submit**, both to catch mistakes and because you should be able to explain your labeling method if asked.


## Step 1 — Setup
Run this in your own new repo's Colab notebook, named `SI26-Week6-Humna.ipynb`.

In [ ]:
import pandas as pd
import re, string, random
from collections import Counter

random.seed(42)


## Step 2 — Collect: download a real Roman Urdu dataset
Source: [`Smat26/Roman-Urdu-Dataset`](https://github.com/Smat26/Roman-Urdu-Dataset) — 20,000+ sentences manually gathered from e-commerce reviews, public Facebook pages, and Twitter, originally compiled by Zareen Sharf (referenced by the UCI Machine Learning Repository). Licensed GPL-3.0 — **credit the original author** if you reuse it (already done below and in the README).

Because this is organic Pakistani social media text, plenty of it is naturally code-switched already — we just need to find and label those sentences.

In [ ]:
!wget -q -O roman_urdu_dataset.csv "https://raw.githubusercontent.com/Smat26/Roman-Urdu-Dataset/master/Dataset/Roman%20Urdu%20DataSet.csv"

raw = pd.read_csv('roman_urdu_dataset.csv', header=None, encoding='utf-8', on_bad_lines='skip')
raw.columns = ['sentence', 'sentiment', 'extra']
raw = raw.drop(columns=['extra'])
raw['sentence'] = raw['sentence'].astype(str).str.strip()
raw = raw.drop_duplicates(subset='sentence')
raw = raw[(raw['sentence'].str.len() > 10) & (raw['sentence'].str.len() < 120)]
print(f'Usable source sentences: {len(raw)}')
raw.sample(5, random_state=1)


## Step 3 — Label: word-by-word URD / ENG / MIX

We need to tell English words from Roman Urdu words. Strategy:
- Download a list of the 10,000 most common English words (`google-10000-english`)
- Maintain a curated list of common Roman Urdu words that happen to *collide* with English spellings (e.g. `"is"`, `"mil"`, `"beta"`, `"par"`) so they don't get mislabeled as English
- Anything left that matches the English list → `ENG`; hyphenated hybrids (like `type-kiya`) → `MIX`; everything else → `URD`

In [ ]:
!wget -q -O google10k.txt "https://raw.githubusercontent.com/first20hours/google-10000-english/master/google-10000-english.txt"

with open('google10k.txt', encoding='utf-8') as f:
    eng_common = set(w.strip().lower() for w in f if w.strip() and w.strip().isalpha())

# Common Roman Urdu words that collide with real English dictionary words
urdu_common = set('''
hai hain ka ki ke ko ny ne se main mai mein maen per pe aur ya na nahi nahin
tha thi thay thy hy hi ho hun hoon kya kia kyun kyu kaisay kaise kab kahan
kidhar kis jis jo woh wo yeh ye ap aap tum tu hum k sy m tak sath saath
agar magar lekin phir ab abhi bhi sirf bohot bahut boht zyada thora thoda
acha achha bura buri nai haan han ji jee sahi galat kar karna kiya karo
karain kre kro hoga hogi honge raha rahi rahe wala wali walay walo liye
lye is us in un uska iska unka apna apni apne mera meri mere tera teri
tere hamara hamari hamare unki unke iski iske uski uske koi kuch sab sub
log logo admi banda larka larki mard aurat ghar din raat waqt baat bat
cheez kaam paisa paisay time dekh dekho suno sun bol bolo chal chalo ja
jao aa aao de do le lo yar yaar bhai behen dost sath kyunke kyunki
taakay taake jab tab jitna utna itna kitna waise aisay aise esa esy
kesy kese karta karti karte hota hoti hote milta milti milte deta deti
dete oar mun ban gae hue gaya gai gaey aya ayi aey shukr shukar h hn
nhi nhe nai bta btao smjh smjho phly phle abi thk theek beta mil das
mar par tou saal aj aaj kal parso hafta mahina sal admi larkay larkiyan
allah masha insha subhanallah alhamdulillah ameen inshallah mashallah
pak rab khuda mn ap aisa waisa itni utni kitni waha yaha idher
udher andar bahar upar neechay agy pechay dour paas yr mary ni b q
'''.split())

def clean_token(tok):
    return tok.strip(string.punctuation).strip()

def label_word(word):
    core = clean_token(word)
    if not core:
        return None
    low = core.lower()
    if '-' in core:
        parts = [p for p in core.split('-') if p]
        eng_part = any(p.lower() in eng_common and p.lower() not in urdu_common for p in parts)
        urd_part = any(p.lower() in urdu_common or p.lower() not in eng_common for p in parts)
        if eng_part and urd_part:
            return 'MIX'
    if low in urdu_common:
        return 'URD'
    if low.isalpha() and low in eng_common:
        return 'ENG'
    return 'URD'

def tokenize_clean(sentence):
    return [clean_token(t) for t in sentence.split() if clean_token(t)]

print('Labeler ready. Example:', list(zip(
    tokenize_clean('Aaj ka din bohot busy tha'),
    [label_word(w) for w in tokenize_clean('Aaj ka din bohot busy tha')]
)))


## Step 3 (cont.) — Filter for genuine code-switching

Not every sentence with one English word counts as "code-switching" — and a few source rows are pure English. We keep a sentence only if:
- it has **2+ English-tagged words** and **3+ Urdu-tagged words** (real bilingual mixing, not a single loanword)
- English words make up **at most ~55%** of the sentence (still recognizably Roman Urdu, not just English)
- no emoji / long digit strings (phone numbers, SMS codes) that aren't real language content

In [ ]:
def has_emoji(s):
    return any(ord(ch) > 10000 for ch in s)

candidates = []
for sent in raw['sentence']:
    if has_emoji(sent) or re.search(r'\d{4,}', sent):
        continue
    words = tokenize_clean(sent)
    if len(words) < 5 or len(words) > 18:
        continue
    if not all(re.match(r"^[A-Za-z']+$", w) for w in words):
        continue

    labels = [label_word(w) for w in words]
    n_eng = labels.count('ENG') + labels.count('MIX')
    n_urd = labels.count('URD')

    if n_eng >= 2 and n_urd >= 3 and n_eng <= len(words) * 0.55:
        candidates.append((' '.join(words), words, labels))

# de-duplicate (case-insensitive) and shuffle for variety
seen, deduped = set(), []
random.shuffle(candidates)
for s, w, l in candidates:
    key = s.lower()
    if key not in seen:
        seen.add(key)
        deduped.append((s, w, l))

print(f'Genuine code-switched candidates found: {len(candidates)}')
print(f'After de-duplication: {len(deduped)}')

TARGET_SENTENCES = 220   # comfortably above the 150+ requirement
data = deduped[:TARGET_SENTENCES]
print(f'Selected: {len(data)} sentences')


## Step 3 (cont.) — Build the labeled dataset
Same output shape as the handout's Cell 1: `sentence`, `word`, `label`.

In [ ]:
rows = []
for sentence, words, labels in data:
    for word, label in zip(words, labels):
        rows.append({
            'sentence': sentence,
            'word': word,
            'label': label
        })

df = pd.DataFrame(rows)
df.to_csv('dataset.csv', index=False, encoding='utf-8')

print(f'Dataset created: {len(df)} word entries')
print(f'Sentences: {df.sentence.nunique()}')
print(f'Label distribution:')
print(df.label.value_counts())
df.head(15)


## Step 5 — Spot-check before you submit (do this by hand)
Pull a random sample and fix anything wrong. This takes ~10 minutes and is the difference between a "silver" auto-labeled set and one you can confidently defend.

In [ ]:
sample_sentences = random.sample(list(df.sentence.unique()), 8)
for s in sample_sentences:
    sub = df[df.sentence == s]
    print(s)
    print(list(zip(sub.word, sub.label)))
    print()

# If you spot a wrong label, fix it directly, e.g.:
# df.loc[(df.sentence == 'exact sentence text') & (df.word == 'word'), 'label'] = 'URD'
# then re-run: df.to_csv('dataset.csv', index=False, encoding='utf-8')


## Step 4 — Publish on Hugging Face

**Manual (recommended, matches the handout):**
1. Go to https://huggingface.co/new-dataset
2. Dataset name: `code-switching-codesaviours-si26-humna`
3. Visibility: **Public**
4. Upload `dataset.csv` (downloaded below)
5. Paste the dataset card text from the next cell into the "Dataset Card" / README

**Or push straight from Colab** (needs a Hugging Face token with write access — `huggingface_hub.login()` will prompt you for it):

In [ ]:
from google.colab import files
files.download('dataset.csv')


In [ ]:
# Optional: push directly from Colab instead of the website upload flow
# from huggingface_hub import login, HfApi
# login()  # paste your HF token (from https://huggingface.co/settings/tokens) when prompted
#
# api = HfApi()
# repo_id = "YOUR_HF_USERNAME/code-switching-codesaviours-si26-humna"
# api.create_repo(repo_id=repo_id, repo_type="dataset", private=False, exist_ok=True)
# api.upload_file(
#     path_or_fileobj="dataset.csv",
#     path_in_repo="dataset.csv",
#     repo_id=repo_id,
#     repo_type="dataset",
# )
# print(f"Uploaded: https://huggingface.co/datasets/{repo_id}")


## Dataset card (paste into the Hugging Face dataset README)

```
---
language:
- ur
- en
license: other
license_name: gpl-3.0-derived-with-additions
tags:
- code-switching
- roman-urdu
- nlp
pretty_name: Code Switching Roman Urdu-English (Code Saviours SI-26)
---

# Code Switching NLP Dataset — Code Saviours SI-26, Humna Imran

Word-level labeled dataset of naturally code-switched Roman Urdu + English
sentences, as commonly written by Pakistani social media users.

## What it is
220 sentences (2,490 word-level rows). Each word is labeled URD (Roman Urdu),
ENG (English), or MIX (hybrid token, e.g. hyphenated compounds).

## How it was built
1. Source sentences come from Smat26/Roman-Urdu-Dataset (GitHub), a public,
   UCI-referenced compilation of ~20,000 Roman Urdu sentences gathered from
   Twitter, Facebook comments, and e-commerce reviews (credit: Zareen Sharf).
2. Sentences were filtered to keep only genuine code-switching: at least
   2 English words and 3 Urdu words, with Urdu still the majority language.
3. Word-level labels were assigned automatically via dictionary lookup
   against the top 10,000 common English words (google-10000-english),
   with a curated override list for Roman Urdu words that collide with
   English spellings (e.g. "mil", "beta", "par"). This mirrors the method
   used in prior Roman-Urdu code-switching research.
4. A random sample was manually spot-checked and corrected.

## Label meanings
- URD: Roman Urdu word
- ENG: English word
- MIX: hybrid/hyphenated token combining both

## Limitations
Automatic dictionary-based labeling can misclassify short words that exist
in both languages by coincidence (e.g. "is", "to"). A curated override list
reduces but does not eliminate this. Treat labels as high-quality silver
annotations, not gold-standard.

## Credit
Source sentences: Smat26/Roman-Urdu-Dataset (GPL-3.0), compiled by Zareen Sharf,
referenced at the UCI Machine Learning Repository.
```
